# Lab 08: Observability & Cost Dashboard

## 🎯 Why This Matters

In Labs 01–07, you built agents, tools, evaluations, and frameworks.
They all **work**. But can you answer these questions?

| Question | Without Observability | With Observability |
|----------|---------------------|--------------------|
| Why is the agent slow? | "I don't know" | "LLM Call #2 took 3.5s — model was overloaded" |
| How much did that cost? | "Check the Azure bill next month" | "$0.021 — 3 LLM calls, 4,200 tokens" |
| What went wrong? | "Run it again and hope" | "Tool `search_knowledge` returned empty at 10:42:03" |
| Who's using the most? | "No idea" | "Tenant 'acme' — 45% of daily spend" |

This lab teaches you **three pillars of observability** for AI agents:

| Part | What | Key Skill |
|------|------|----------|
| **Part 1** | OpenTelemetry Tracing | Instrument a LangGraph agent end-to-end |
| **Part 2** | Token & Cost Tracking | Build a live cost dashboard |
| **Part 3** | Custom Callback Handler | Structured logging for every agent step |

> **Prerequisite:** Read [Chapter 11: Observability & Cost](../../education/en/11-observability-cost.md)

---

## 🔧 Setup

We install OpenTelemetry, LangGraph, and connect to the same Azure OpenAI from Lab 01.

In [ ]:
# ──────────────────────────────────────────────────────
# Install packages
#
# • opentelemetry-sdk + api — Core tracing framework
# • opentelemetry-exporter-otlp — Export traces (optional)
# • langgraph + langchain-openai — The agent we'll instrument
# • python-dotenv — Load Azure credentials from .env
# • rich — Pretty output for traces
# ──────────────────────────────────────────────────────
%pip install opentelemetry-sdk opentelemetry-api --quiet
%pip install langgraph langchain-openai python-dotenv rich --quiet

In [ ]:
import os
import time
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv("../.env")

AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")

print(f"✅ Azure OpenAI endpoint: {AZURE_ENDPOINT}")
print(f"   Deployment: {MODEL}")

### Shared Tools (same as Lab 01 and Lab 07)

We reuse the exact same tools so you can compare the observable agent to the raw one.

In [ ]:
# ──────────────────────────────────────────────────────
# Shared tools — identical to Lab 01 & Lab 07
# ──────────────────────────────────────────────────────

def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {
        "tel aviv": {"temp": 28, "condition": "sunny", "humidity": 65},
        "new york": {"temp": 12, "condition": "cloudy", "humidity": 70},
        "london":   {"temp": 8,  "condition": "rainy",  "humidity": 85},
        "tokyo":    {"temp": 20, "condition": "clear",  "humidity": 55},
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "unknown", "humidity": 50})
    return f"Weather in {city}: {data['temp']}°C, {data['condition']}, humidity {data['humidity']}%"


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Only supports basic arithmetic."""
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return "Error: only mathematical expressions are allowed"
    try:
        result = eval(expression)
        return f"Result: {expression} = {result}"
    except Exception as e:
        return f"Error calculating: {e}"


KNOWLEDGE_BASE = [
    {"topic": "vacation policy", "content": "Employees get 22 vacation days per year. Unused days can carry over up to 5 days."},
    {"topic": "remote work", "content": "Employees can work remotely up to 3 days per week. Core hours 10am-4pm."},
    {"topic": "expense policy", "content": "Meals up to $50/day during travel. Flights must be economy. Hotel max $200/night."},
    {"topic": "onboarding", "content": "New employees have a 90-day onboarding. Week 1: HR orientation. Month 2-3: projects."},
]

def search_knowledge(query: str) -> str:
    """Search the company knowledge base for policies and procedures."""
    query_lower = query.lower()
    results = []
    for item in KNOWLEDGE_BASE:
        if any(word in query_lower for word in item["topic"].split()):
            results.append(f"[{item['topic']}]: {item['content']}")
    return "\n\n".join(results) if results else "No relevant information found."


print("✅ All 3 shared tools ready")

---

# Part 1: OpenTelemetry Tracing

## What is OpenTelemetry?

**OpenTelemetry (OTel)** is the industry standard for instrumentation. It lets you:
- Create **spans** — timed operations ("LLM call took 1.2s")
- Build **traces** — trees of spans showing the full request flow
- Add **attributes** — metadata (model name, token count, tool name)
- Export to **any backend** — Azure Monitor, Jaeger, Grafana, console

### Why not just use `print()`?

| `print()` Debugging | OpenTelemetry |
|---------------------|---------------|
| Scattered across code | Centralized, structured |
| No timing | Automatic duration per span |
| No relationships | Parent→child span trees |
| Text only | Rich attributes (numbers, lists) |
| Console only | Export to any dashboard |
| Remove before production | Keep in production — zero cost when disabled |

### The Key Concepts

```
Trace (the full request)
├── Span: "agent_request"              ← parent span
│   ├── Span: "llm_call_1"             ← child: first LLM thinking step
│   ├── Span: "tool_call: get_weather"  ← child: tool execution
│   ├── Span: "llm_call_2"             ← child: LLM observes tool result
│   └── Span: "llm_call_3"             ← child: final answer
```

Each span has:
- **Name** — what operation is this
- **Start/End time** — how long it took
- **Attributes** — metadata (model, tokens, cost)
- **Status** — OK or ERROR
- **Parent** — which span created this one

## Step 1: Set Up the Tracer Provider

The `TracerProvider` is the entry point for all tracing. We configure it with:
- A **Resource** — identifies our service
- A **SpanProcessor** — where to send completed spans

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource

# ──────────────────────────────────────────────────────
# Custom In-Memory Exporter
#
# In production, you'd export to Azure Monitor, Jaeger,
# or Grafana Tempo. For this lab, we collect spans in
# memory so we can analyze and visualize them ourselves.
# ──────────────────────────────────────────────────────
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

class InMemorySpanExporter(SpanExporter):
    """Collects spans in memory for analysis."""
    def __init__(self):
        self.spans = []
    
    def export(self, spans):
        self.spans.extend(spans)
        return SpanExportResult.SUCCESS
    
    def shutdown(self):
        pass
    
    def clear(self):
        self.spans = []

# Create the exporter and tracer provider
span_exporter = InMemorySpanExporter()

resource = Resource.create({
    "service.name": "lab08-agent-platform",
    "service.version": "1.0.0",
})

provider = TracerProvider(resource=resource)
provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("lab08.agent")

print("✅ OpenTelemetry tracer provider configured")
print("   Spans will be collected in memory for analysis")

## Step 2: Instrument a LangGraph Agent

Now we create the **same LangGraph agent from Lab 01/07**, but this time we:
1. Wrap tool functions to emit spans on every call
2. Use a custom callback handler to trace LLM calls
3. Create a parent span for the entire agent request

This is what real production instrumentation looks like.

In [ ]:
from langchain_openai import AzureChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from opentelemetry.trace import StatusCode

# ──────────────────────────────────────────────────────
# Step 2a: Create instrumented tool wrappers
#
# These wrap our plain functions with OpenTelemetry spans.
# Every tool call will now appear as a span with:
# - tool name, arguments, duration
# - result (truncated for safety)
# - error status if the tool fails
# ──────────────────────────────────────────────────────

@tool
def traced_get_weather(city: str) -> str:
    """Get the current weather for a city."""
    with tracer.start_as_current_span("tool_call") as span:
        span.set_attribute("tool.name", "get_weather")
        span.set_attribute("tool.args.city", city)
        result = get_weather(city)
        span.set_attribute("tool.result", result[:200])
        return result

@tool
def traced_calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    with tracer.start_as_current_span("tool_call") as span:
        span.set_attribute("tool.name", "calculate")
        span.set_attribute("tool.args.expression", expression)
        result = calculate(expression)
        span.set_attribute("tool.result", result[:200])
        return result

@tool
def traced_search_knowledge(query: str) -> str:
    """Search the company knowledge base for policies and procedures."""
    with tracer.start_as_current_span("tool_call") as span:
        span.set_attribute("tool.name", "search_knowledge")
        span.set_attribute("tool.args.query", query)
        result = search_knowledge(query)
        span.set_attribute("tool.result", result[:200])
        return result

tools = [traced_get_weather, traced_calculate, traced_search_knowledge]
print("✅ Instrumented tools created")

In [ ]:
# ──────────────────────────────────────────────────────
# Step 2b: Create the LangGraph agent
#
# Same agent as Lab 01/07 — but now with traced tools.
# ──────────────────────────────────────────────────────

llm = AzureChatOpenAI(
    azure_deployment=MODEL,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
)

SYSTEM_PROMPT = (
    "You are a helpful assistant for Acme Corp employees. "
    "Use the available tools to answer questions about weather, "
    "company policies, and calculations."
)

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print(f"✅ LangGraph agent created with {len(tools)} instrumented tools")

## Step 3: Run the Agent with Tracing

We wrap the entire agent invocation in a **parent span**. All tool spans
become children of this parent, giving us a full trace tree.

In [ ]:
# ──────────────────────────────────────────────────────
# Run the agent with a parent trace span.
#
# This is the same question from Lab 01/07 — but now
# every step is being traced.
# ──────────────────────────────────────────────────────

TEST_QUESTION = (
    "I'm planning a trip to Tokyo. What's the weather there? "
    "Also, what's the daily meal expense limit? "
    "If I eat 3 meals at $15 each, am I within budget?"
)

span_exporter.clear()  # Clear any previous spans

with tracer.start_as_current_span("agent_request") as parent_span:
    parent_span.set_attribute("agent.name", "acme-assistant")
    parent_span.set_attribute("agent.model", MODEL)
    parent_span.set_attribute("user.question", TEST_QUESTION[:200])
    
    start = time.time()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": TEST_QUESTION}]}
    )
    duration = time.time() - start
    
    answer = result["messages"][-1].content
    parent_span.set_attribute("agent.duration_s", round(duration, 2))
    parent_span.set_attribute("agent.answer_length", len(answer))

print(f"✅ Agent completed in {duration:.1f}s")
print(f"\n📤 Answer:\n{answer[:500]}")

## Step 4: Visualize the Trace

Now let's look at what was captured. We'll build a **trace timeline** — the same
view you'd see in Jaeger, Azure Monitor, or any tracing UI.

This is the "aha" moment — you can see exactly where time was spent.

In [ ]:
# ──────────────────────────────────────────────────────
# Visualize the trace as a timeline.
#
# This recreates the "waterfall" view from tools like
# Jaeger, Zipkin, or Azure Monitor — but in the terminal.
# ──────────────────────────────────────────────────────

spans = span_exporter.spans
print("=" * 80)
print("🔗 TRACE TIMELINE")
print("=" * 80)
print(f"\nTotal spans captured: {len(spans)}")

if spans:
    # Find the earliest start time for relative timing
    min_start = min(s.start_time for s in spans)
    max_end = max(s.end_time for s in spans)
    total_ns = max_end - min_start
    total_ms = total_ns / 1_000_000
    
    print(f"Total trace duration: {total_ms:.0f}ms\n")
    print(f"{'Span Name':<35} {'Duration':>10} {'Timeline (50 chars)':>52}")
    print("─" * 100)
    
    for span in spans:
        name = span.name
        dur_ns = span.end_time - span.start_time
        dur_ms = dur_ns / 1_000_000
        
        # Calculate position in timeline bar
        bar_width = 50
        start_pos = int(((span.start_time - min_start) / total_ns) * bar_width) if total_ns > 0 else 0
        end_pos = int(((span.end_time - min_start) / total_ns) * bar_width) if total_ns > 0 else bar_width
        span_width = max(1, end_pos - start_pos)
        
        bar = "░" * start_pos + "▓" * span_width + "░" * (bar_width - start_pos - span_width)
        
        # Add tool name if it's a tool call
        tool_name = ""
        attrs = dict(span.attributes) if span.attributes else {}
        if "tool.name" in attrs:
            tool_name = f" [{attrs['tool.name']}]"
        
        print(f"  {name + tool_name:<33} {dur_ms:>8.0f}ms  {bar}")

print("\n" + "=" * 80)

## Step 5: Inspect Span Attributes

Each span carries rich metadata. Let's inspect what was captured
— tool names, arguments, results, and more.

In [ ]:
# ──────────────────────────────────────────────────────
# Inspect span attributes — the detailed metadata
# that makes traces useful for debugging.
# ──────────────────────────────────────────────────────

print("=" * 80)
print("📋 SPAN DETAILS")
print("=" * 80)

for i, span in enumerate(spans):
    dur_ms = (span.end_time - span.start_time) / 1_000_000
    attrs = dict(span.attributes) if span.attributes else {}
    
    print(f"\n[{i+1}] {span.name}")
    print(f"    Duration: {dur_ms:.0f}ms")
    print(f"    Status:   {span.status.status_code.name}")
    
    if attrs:
        print(f"    Attributes:")
        for key, value in attrs.items():
            # Truncate long values
            val_str = str(value)
            if len(val_str) > 80:
                val_str = val_str[:77] + "..."
            print(f"      {key}: {val_str}")

print("\n" + "=" * 80)

## 📊 Part 1 Summary

| Concept | What It Is | Why It Matters |
|---------|-----------|----------------|
| **Trace** | The full request end-to-end | See the big picture |
| **Span** | A single operation (LLM call, tool call) | Pinpoint bottlenecks |
| **Attributes** | Metadata on each span | Debug with context |
| **Parent-Child** | Span hierarchy | Understand call flow |
| **Exporter** | Where spans go (console, Jaeger, Azure Monitor) | Production-ready |

In production, you'd replace `InMemorySpanExporter` with:
- `AzureMonitorTraceExporter` → Azure Monitor / Application Insights
- `OTLPSpanExporter` → Jaeger, Grafana Tempo, or any OTLP backend

---

# Part 2: Token & Cost Tracking

## Why Token Tracking?

**Tokens = money.** Every LLM call costs money, and costs can spiral fast:

| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|-------|----------------------|------------------------|
| GPT-4.1 | $2.00 | $8.00 |
| GPT-4o | $2.50 | $10.00 |
| GPT-4o-mini | $0.15 | $0.60 |

A single agent request might make 3-5 LLM calls. Multiply by 1,000 users and
you need to track every token.

### Real-World Horror Story

```
Friday 5 PM:  Team deploys a prompt change → goes home
Saturday:     Bug causes agent to retry LLM calls in a loop
Sunday:       500,000 API calls executed
Monday 9 AM:  Azure bill: $10,000 over the weekend

WITH token tracking + alerts:
Friday 6 PM:  Token rate alert fires (10x normal)
Friday 6:01:  Auto-kill triggered (budget exceeded)
Total damage: $50 instead of $10,000
```

## Step 6: Build a Token Counter

We build a middleware that wraps every LLM call and extracts token usage
from the API response. This is the foundation of cost tracking.

In [ ]:
# ──────────────────────────────────────────────────────
# Token & Cost Tracker
#
# This class records every LLM call's token usage
# and calculates the cost based on the model's pricing.
#
# In production, you'd write these records to a
# time-series database (InfluxDB, Azure Monitor, etc.)
# ──────────────────────────────────────────────────────

class CostTracker:
    """Tracks token usage and costs across agent requests."""
    
    # Pricing per 1M tokens (Azure OpenAI, as of 2025)
    PRICING = {
        "gpt-41":       {"input": 2.00, "output": 8.00},
        "gpt-4o":       {"input": 2.50, "output": 10.00},
        "gpt-4o-mini":  {"input": 0.15, "output": 0.60},
    }
    
    def __init__(self):
        self.records = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0.0
    
    def record(self, model: str, input_tokens: int, output_tokens: int,
               agent_name: str = "default", request_id: str = ""):
        """Record a single LLM call."""
        pricing = self.PRICING.get(model, {"input": 2.50, "output": 10.00})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1_000_000
        
        record = {
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "cost_usd": round(cost, 6),
            "agent_name": agent_name,
            "request_id": request_id,
        }
        self.records.append(record)
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_cost += cost
        return record
    
    def summary(self) -> dict:
        """Return an aggregated summary."""
        return {
            "total_calls": len(self.records),
            "total_input_tokens": self.total_input_tokens,
            "total_output_tokens": self.total_output_tokens,
            "total_tokens": self.total_input_tokens + self.total_output_tokens,
            "total_cost_usd": round(self.total_cost, 4),
        }
    
    def by_model(self) -> dict:
        """Aggregate costs by model."""
        models = {}
        for r in self.records:
            m = r["model"]
            if m not in models:
                models[m] = {"calls": 0, "tokens": 0, "cost": 0.0}
            models[m]["calls"] += 1
            models[m]["tokens"] += r["total_tokens"]
            models[m]["cost"] += r["cost_usd"]
        return models
    
    def by_agent(self) -> dict:
        """Aggregate costs by agent."""
        agents = {}
        for r in self.records:
            a = r["agent_name"]
            if a not in agents:
                agents[a] = {"calls": 0, "tokens": 0, "cost": 0.0}
            agents[a]["calls"] += 1
            agents[a]["tokens"] += r["total_tokens"]
            agents[a]["cost"] += r["cost_usd"]
        return agents

cost_tracker = CostTracker()
print("✅ Cost tracker initialized")
print(f"   Models configured: {list(CostTracker.PRICING.keys())}")

## Step 7: Run Multiple Agent Requests & Track Costs

Let's run several agent requests with different models and track token usage.
We extract usage metadata from the LangGraph response messages.

In [ ]:
# ──────────────────────────────────────────────────────
# Run multiple requests and track costs.
#
# LangChain's AIMessage includes usage_metadata when
# the model returns it. We extract tokens from there.
# ──────────────────────────────────────────────────────

questions = [
    ("weather-agent", "What's the weather in Tokyo and London?"),
    ("policy-agent",  "What's our travel expense policy for meals and hotels?"),
    ("math-agent",    "If I have 3 meetings at $15 coffee each for 5 days, what's the total?"),
    ("combo-agent",   TEST_QUESTION),  # The multi-tool question
]

print("📊 Running agent requests and tracking costs...\n")

for agent_name, question in questions:
    with tracer.start_as_current_span("agent_request") as span:
        span.set_attribute("agent.name", agent_name)
        
        start = time.time()
        result = agent.invoke(
            {"messages": [{"role": "user", "content": question}]}
        )
        dur = time.time() - start
        
        # Extract token usage from LLM response messages
        for msg in result["messages"]:
            usage = getattr(msg, "usage_metadata", None)
            if usage:
                input_t = usage.get("input_tokens", 0)
                output_t = usage.get("output_tokens", 0)
                rec = cost_tracker.record(
                    model=MODEL,
                    input_tokens=input_t,
                    output_tokens=output_t,
                    agent_name=agent_name,
                )
        
        print(f"  ✅ {agent_name:<15} {dur:.1f}s  Q: {question[:50]}...")

print(f"\n📊 Total LLM calls tracked: {len(cost_tracker.records)}")

## Step 8: Cost Dashboard

Now let's build the **live cost dashboard** from Chapter 11.
This is the view that every production agent platform needs.

In [ ]:
# ──────────────────────────────────────────────────────
# Cost Dashboard — the same layout from Ch. 11
# ──────────────────────────────────────────────────────

summary = cost_tracker.summary()
by_model = cost_tracker.by_model()
by_agent = cost_tracker.by_agent()

print("┌" + "─" * 60 + "┐")
print("│" + "💰 COST OBSERVABILITY DASHBOARD".center(60) + "│")
print("├" + "─" * 60 + "┤")
print("│" + "│")
print(f"│  📊 Total LLM Calls:     {summary['total_calls']:<30}  │")
print(f"│  📥 Total Input Tokens:   {summary['total_input_tokens']:<30,}  │")
print(f"│  📤 Total Output Tokens:  {summary['total_output_tokens']:<30,}  │")
print(f"│  🔢 Total Tokens:         {summary['total_tokens']:<30,}  │")
print(f"│  💵 Total Cost:           ${summary['total_cost_usd']:<29}  │")
print("│" + "│")
print("├" + "─" * 60 + "┤")
print("│" + " 🧠 Cost by Model".ljust(60) + "│")
print("├" + "─" * 60 + "┤")

for model_name, data in by_model.items():
    print(f"│  {model_name:<15} {data['calls']:>3} calls  {data['tokens']:>8,} tokens  ${data['cost']:.4f}  │")

print("├" + "─" * 60 + "┤")
print("│" + " 🤖 Cost by Agent".ljust(60) + "│")
print("├" + "─" * 60 + "┤")

for a_name, data in sorted(by_agent.items(), key=lambda x: -x[1]["cost"]):
    bar_len = int(data['cost'] / max(d['cost'] for d in by_agent.values()) * 20) if by_agent else 0
    bar = "█" * max(1, bar_len)
    print(f"│  {a_name:<15} ${data['cost']:.4f}  {bar:<20}  {data['calls']} calls  │")

print("├" + "─" * 60 + "┤")
print("│" + " 📋 Per-Call Breakdown".ljust(60) + "│")
print("├" + "─" * 60 + "┤")
print(f"│  {'Agent':<15} {'In':>6} {'Out':>6} {'Total':>7} {'Cost':>9}   │")
print("│  " + "─" * 46 + "             │")

for r in cost_tracker.records:
    print(f"│  {r['agent_name']:<15} {r['input_tokens']:>6,} {r['output_tokens']:>6,} {r['total_tokens']:>7,} ${r['cost_usd']:.6f}   │")

print("└" + "─" * 60 + "┘")

## Step 9: Budget Alerts

In production, you need **automatic alerts** when costs exceed thresholds.
This prevents the "$10,000 weekend" scenario from Chapter 11.

In [ ]:
# ──────────────────────────────────────────────────────
# Budget Alert System
#
# In production, these would fire Slack/PagerDuty alerts.
# Here we demonstrate the logic.
# ──────────────────────────────────────────────────────

class BudgetAlertSystem:
    """Checks cost tracker against budget thresholds."""
    
    def __init__(self, daily_budget: float, per_request_max: float, token_rate_max: int):
        self.daily_budget = daily_budget
        self.per_request_max = per_request_max
        self.token_rate_max = token_rate_max
    
    def check(self, tracker: CostTracker) -> list:
        """Return list of triggered alerts."""
        alerts = []
        summary = tracker.summary()
        
        # Budget percentage alert
        pct = (summary["total_cost_usd"] / self.daily_budget) * 100
        if pct >= 90:
            alerts.append(("🔴 P1", f"Daily budget at {pct:.0f}% (${summary['total_cost_usd']:.4f} / ${self.daily_budget})"))
        elif pct >= 70:
            alerts.append(("🟡 P2", f"Daily budget at {pct:.0f}% (${summary['total_cost_usd']:.4f} / ${self.daily_budget})"))
        
        # Per-request cost spike
        for r in tracker.records:
            if r["cost_usd"] > self.per_request_max:
                alerts.append(("🟡 P2", f"Request cost spike: ${r['cost_usd']:.4f} > ${self.per_request_max} (agent: {r['agent_name']})"))
        
        # Total token rate
        if summary["total_tokens"] > self.token_rate_max:
            alerts.append(("🟠 P2", f"Token usage high: {summary['total_tokens']:,} > {self.token_rate_max:,} limit"))
        
        if not alerts:
            alerts.append(("✅ OK", "All metrics within normal range"))
        
        return alerts

# Create alert system with budgets
alert_system = BudgetAlertSystem(
    daily_budget=1.00,        # $1/day budget for demo
    per_request_max=0.05,     # Max $0.05 per request
    token_rate_max=50_000,    # Max 50K tokens before alert
)

# Check alerts
alerts = alert_system.check(cost_tracker)

print("=" * 60)
print("🚨 ALERT STATUS")
print("=" * 60)
for severity, message in alerts:
    print(f"  {severity}  {message}")
print("=" * 60)

## 📊 Part 2 Summary

| What We Built | Production Equivalent |
|--------------|----------------------|
| `CostTracker` class | InfluxDB / Azure Monitor custom metrics |
| `by_model()` / `by_agent()` | Grafana dashboard dimensions |
| ASCII dashboard | Grafana / Azure Dashboard |
| `BudgetAlertSystem` | Azure Monitor Alerts / PagerDuty |

---

# Part 3: Custom Callback Handler

## Why Callbacks?

LangGraph/LangChain provides **callback hooks** that fire at every step:
- `on_llm_start` / `on_llm_end` — when an LLM call starts/finishes
- `on_tool_start` / `on_tool_end` — when a tool is called
- `on_chain_start` / `on_chain_end` — when a chain/graph step runs

By writing a custom callback handler, you can capture **structured logs**
for every single operation — without modifying agent code.

### Callback vs Manual Tracing

| Manual (Part 1) | Callbacks (Part 3) |
|-----------------|--------------------|
| You add spans to each tool | Framework adds callbacks automatically |
| You wrap each function | Write one handler, instrument everything |
| Fine-grained control | Framework-level coverage |
| More code | Less code |

In production, you use **both**: callbacks for broad coverage + manual spans for
critical business logic.

## Step 10: Build a Custom Callback Handler

This handler captures every LLM call and tool call with structured data.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from typing import Any

class AgentObservabilityHandler(BaseCallbackHandler):
    """Custom callback handler that logs every agent step."""
    
    def __init__(self, cost_tracker: CostTracker, agent_name: str = "default"):
        self.cost_tracker = cost_tracker
        self.agent_name = agent_name
        self.events = []
        self._llm_starts = {}  # Track start times by run_id
    
    def on_llm_start(self, serialized: dict, prompts: list, *, run_id, **kwargs):
        self._llm_starts[str(run_id)] = time.time()
        self.events.append({
            "type": "llm_start",
            "timestamp": datetime.now().isoformat(),
            "run_id": str(run_id),
            "model": serialized.get("kwargs", {}).get("model_name", "unknown"),
        })
    
    def on_llm_end(self, response, *, run_id, **kwargs):
        duration = time.time() - self._llm_starts.get(str(run_id), time.time())
        
        # Extract token usage
        llm_output = response.llm_output or {}
        usage = llm_output.get("token_usage", {})
        input_t = usage.get("prompt_tokens", 0)
        output_t = usage.get("completion_tokens", 0)
        
        if input_t > 0 or output_t > 0:
            self.cost_tracker.record(
                model=MODEL,
                input_tokens=input_t,
                output_tokens=output_t,
                agent_name=self.agent_name,
            )
        
        self.events.append({
            "type": "llm_end",
            "timestamp": datetime.now().isoformat(),
            "run_id": str(run_id),
            "duration_s": round(duration, 2),
            "input_tokens": input_t,
            "output_tokens": output_t,
        })
    
    def on_tool_start(self, serialized: dict, input_str: str, *, run_id, **kwargs):
        self._llm_starts[str(run_id)] = time.time()
        tool_name = serialized.get("name", "unknown")
        self.events.append({
            "type": "tool_start",
            "timestamp": datetime.now().isoformat(),
            "tool": tool_name,
            "input": str(input_str)[:100],
        })
    
    def on_tool_end(self, output: str, *, run_id, **kwargs):
        duration = time.time() - self._llm_starts.get(str(run_id), time.time())
        self.events.append({
            "type": "tool_end",
            "timestamp": datetime.now().isoformat(),
            "duration_s": round(duration, 3),
            "output": str(output)[:100],
        })

print("✅ AgentObservabilityHandler defined")

## Step 11: Run the Agent with the Callback

We pass our custom handler in the `config` parameter. LangGraph will call
our handler at every step — **zero changes to agent code**.

In [ ]:
# ──────────────────────────────────────────────────────
# Run the agent with our callback handler.
#
# Notice: we don't change the agent at all.
# We just pass the callback in the config.
# ──────────────────────────────────────────────────────

# Reset the cost tracker for a clean demo
callback_tracker = CostTracker()
handler = AgentObservabilityHandler(
    cost_tracker=callback_tracker,
    agent_name="callback-demo-agent"
)

callback_result = agent.invoke(
    {"messages": [{"role": "user", "content": TEST_QUESTION}]},
    config={"callbacks": [handler]},
)

answer = callback_result["messages"][-1].content
print(f"📤 Answer: {answer[:300]}...\n")

In [ ]:
# ──────────────────────────────────────────────────────
# Display the structured event log
# ──────────────────────────────────────────────────────

print("=" * 80)
print("📋 STRUCTURED EVENT LOG")
print("=" * 80)

for i, event in enumerate(handler.events, 1):
    event_type = event["type"]
    
    if event_type == "llm_start":
        print(f"\n  [{i}] 🧠 LLM Start")
    elif event_type == "llm_end":
        dur = event.get('duration_s', '?')
        inp = event.get('input_tokens', 0)
        out = event.get('output_tokens', 0)
        print(f"  [{i}] 🧠 LLM End — {dur}s, {inp} in + {out} out tokens")
    elif event_type == "tool_start":
        tool = event.get('tool', 'unknown')
        inp = event.get('input', '')[:60]
        print(f"  [{i}] 🔧 Tool Start: {tool}({inp})")
    elif event_type == "tool_end":
        dur = event.get('duration_s', '?')
        out = event.get('output', '')[:60]
        print(f"  [{i}] 🔧 Tool End — {dur}s → {out}")

print(f"\n  Total events: {len(handler.events)}")
print("\n" + "=" * 80)

## Step 12: Multi-Turn Cost Tracking

In real applications, users have multi-turn conversations. Each turn costs tokens.
Let's track how costs **grow** across a conversation — the same pattern
that causes budget overruns in production.

In [ ]:
# ──────────────────────────────────────────────────────
# Multi-turn conversation with cumulative cost tracking.
#
# Watch how costs grow with each turn — because chat
# history gets longer, more input tokens are needed.
# This is why memory management (Lab 03) is critical!
# ──────────────────────────────────────────────────────

from langgraph.checkpoint.memory import MemorySaver

# Create agent with memory (checkpointing)
agent_with_memory = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

conversation_tracker = CostTracker()
conv_handler = AgentObservabilityHandler(
    cost_tracker=conversation_tracker,
    agent_name="multi-turn-agent"
)

config = {
    "configurable": {"thread_id": "cost-demo-thread"},
    "callbacks": [conv_handler],
}

turns = [
    "What's the weather in Tokyo?",
    "And what about London?",
    "Compare both cities — which is better for a trip?",
    "What's the expense policy for travel meals?",
    "If I eat 3 meals at $15 each per day for 5 days, what does that cost?",
]

print("=" * 70)
print("📊 MULTI-TURN COST GROWTH")
print("=" * 70)
print(f"{'Turn':<5} {'Cumulative Cost':>15} {'Turn Tokens':>13} {'Total Tokens':>14} {'Question'}")
print("─" * 70)

prev_cost = 0
prev_tokens = 0

for i, question in enumerate(turns, 1):
    agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    
    s = conversation_tracker.summary()
    turn_tokens = s["total_tokens"] - prev_tokens
    turn_cost = s["total_cost_usd"] - prev_cost
    
    bar_len = min(20, int(turn_tokens / 200))
    bar = "█" * max(1, bar_len)
    
    print(f"  {i:<3}  ${s['total_cost_usd']:>12.4f}   {turn_tokens:>10,}   {s['total_tokens']:>11,}   {question[:35]}")
    prev_cost = s["total_cost_usd"]
    prev_tokens = s["total_tokens"]

print("─" * 70)
print(f"\n💡 Notice: later turns cost MORE tokens because chat history grows!")
print(f"   This is why memory management (Lab 03) and summarization matter.")
print("\n" + "=" * 70)

## 📊 Part 3 Summary

| What We Built | Production Equivalent |
|--------------|----------------------|
| `AgentObservabilityHandler` | LangSmith / Datadog LLM callback |
| Structured event log | ELK Stack / Azure Log Analytics |
| Multi-turn cost growth | Budget monitoring per conversation |

---

## 🧹 Cleanup

In [ ]:
# Shut down the tracer provider to flush any remaining spans
tp = trace.get_tracer_provider()
if hasattr(tp, 'shutdown'):
    tp.shutdown()
    print("✅ Tracer shut down")

print("✨ Lab 08 complete!")

---

## 🎓 What You've Learned

### Three Pillars, Applied

| Pillar | What You Built | Production Tool |
|--------|---------------|------------------|
| **Traces** | OpenTelemetry spans with timeline visualization | Jaeger, Azure Monitor, Grafana Tempo |
| **Metrics** | Token counter, cost aggregation, budget alerts | Prometheus, Azure Monitor Metrics |
| **Logs** | Structured callback event log | ELK Stack, Azure Log Analytics |

### Key Takeaways

1. **Traces show WHERE time is spent** — which LLM call, which tool, which step
2. **Token tracking shows HOW MUCH it costs** — per model, per agent, per request
3. **Callbacks give you AUTOMATIC coverage** — every step logged without changing agent code
4. **Multi-turn costs GROW** — because chat history = more input tokens per turn
5. **Budget alerts PREVENT disasters** — $50 alert vs $10,000 weekend

### What's Next?

In **Lab 09**, you'll see how Azure AI Foundry provides **all of this out of the box** —
built-in tracing, evaluations, and monitoring — no manual instrumentation needed.

---

> **[← Lab 07: Framework Deep Dive](../lab-07-frameworks/README.md)** | **[→ Lab 09: Azure AI Foundry](../lab-09-foundry/README.md)**